Synthetic discharge summary PDFs are in the `data/` folder. All patient data is fabricated — no real health records are used.

Set the three variables in the config cell before running.

   
## ai_parse_document on Databricks - Companion Notebook
This notebook accompanies the blog post *ai_parse_document on Databricks Looks Like Magic. Here's What to Solve Before Production.*

It demonstrates a Bronze → Silver medallion pattern for extracting structured fields from synthetic hospital discharge summaries using `ai_parse_document` and `ai_query`. This is a proof-of-concept, not a production pipeline.

**Requirements:** Databricks Runtime 15.4+, Unity Catalog, a model serving endpoint with access to a Claude or GPT-4 class model.

In [0]:
# === CONFIGURE BEFORE RUNNING ===
PDF_PATH = "/path/to/your/pdfs"  # path to your PDFs in DBFS or a Volume
CATALOG  = "your_catalog"
SCHEMA   = "healthcare_demo"

In [0]:
# Expose config to SQL cells
dbutils.widgets.text("pdf_path", PDF_PATH)
dbutils.widgets.text("catalog", CATALOG)
dbutils.widgets.text("schema", SCHEMA)

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

import hashlib
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Demo pseudonymisation - production should use SHA-256 + Azure Key Vault salt
def pseudonymise(value: str) -> str:
    if not value:
        return None
    return hashlib.md5(f"DEMO_SALT_{value}".encode()).hexdigest()

spark.udf.register("pseudonymise", pseudonymise, StringType())

<function __main__.pseudonymise(value: str) -> str>

   
### Loading PDFs into Spark

`ai_parse_document` expects a `BINARY` column. How you get there depends on your setup:

* **Option A** - You have a UC Volume or a classic cluster. Run the SQL cell below.
* **Option B** - You're on serverless with workspace files. Skip Option A and run the Python cell instead.

Both produce the same `raw_pdfs` temp view. Run **one**, not both.

In [0]:
%sql
-- Option A: UC Volume or classic cluster

-- CREATE OR REPLACE TEMP VIEW raw_pdfs AS
-- SELECT
--     substring_index(path, '/', -1) AS filename,
--     content
-- FROM read_files(
--     :pdf_path,
--     format => 'binaryFile',
--     pathGlobFilter => '*.pdf'
-- );

In [0]:
# Option B: Serverless workaround

import os
from pyspark.sql.types import StructType, StructField, StringType, BinaryType

rows = [(f, open(os.path.join(PDF_PATH, f), "rb").read())
        for f in sorted(os.listdir(PDF_PATH)) if f.endswith(".pdf")]

raw_pdfs = spark.createDataFrame(rows, StructType([
    StructField("filename", StringType()),
    StructField("content", BinaryType())
]))
raw_pdfs.createOrReplaceTempView("raw_pdfs")

print(f"Loaded {len(rows)} PDFs into raw_pdfs temp view")

Loaded 65 PDFs into raw_pdfs temp view


In [0]:
%sql
    
-- Proof of Concept: ai_parse_document + ai_query in one shot
-- Note: patient_ref is in plain text here - pseudonymisation is added in Bronze

WITH raw AS (
    SELECT
        filename,
        ai_parse_document(content, map('version', '2.0')) AS parsed
    FROM raw_pdfs
    LIMIT 1
),
text_extracted AS (
    SELECT
        filename,
        concat_ws('\n', transform(
            try_cast(parsed:document:elements AS ARRAY<VARIANT>),
            element -> try_cast(element:content AS STRING)
        )) AS full_text
    FROM raw
    WHERE try_cast(parsed:error_status AS STRING) IS NULL
)
SELECT
    filename,
    ai_query(
        'databricks-meta-llama-3-1-70b-instruct',
        concat(
            'Extract as JSON: {"patient_ref": "string", "document_date": "YYYY-MM-DD", ',
            '"document_type": "discharge_summary|referral_letter|admission_note", ',
            '"primary_diagnosis": "string", "follow_up_required": true|false}. ',
            'Text may be Dutch, Latin, or English.\n\nDocument:\n', full_text
        ),
        returnType => 'STRING'
    ) AS extracted
FROM text_extracted;

filename,extracted
admission_PT_2024_00502_digital.pdf,"Here is the extracted data in JSON format: ```json { ""patient_ref"": ""PT-2024-00502"", ""document_date"": ""2024-03-28"", ""document_type"": ""admission_note"", ""primary_diagnosis"": ""Subarachnoid haemorrhage"", ""follow_up_required"": true } ``` Note: The `follow_up_required` field is set to `true` based on the fact that the patient is being admitted to the hospital and a management plan is being put in place, which typically involves follow-up care. However, this field is not explicitly stated in the document, so this is an inference based on the context."


In [0]:
# Bronze: Parse PDFs, pseudonymise, deduplicate by content hash

bronze_df = spark.sql("""
WITH raw AS (
    SELECT
        filename,
        md5(content) AS content_hash,
        ai_parse_document(content, map('version', '2.0')) AS parsed
    FROM raw_pdfs
    ORDER BY rand()
    LIMIT 5
),
text_extracted AS (
    SELECT
        filename,
        content_hash,
        concat_ws('\\n', transform(
            try_cast(parsed:document:elements AS ARRAY<VARIANT>),
            element -> try_cast(element:content AS STRING)
        )) AS full_text
    FROM raw
    WHERE try_cast(parsed:error_status AS STRING) IS NULL
),
ai_extraction AS (
    SELECT
        filename,
        content_hash,
        ai_query(
            'databricks-meta-llama-3-1-70b-instruct',
            concat(
                'Extract ONLY the JSON object with these exact fields: ',
                '{{"patient_ref": "patient ID", "document_date": "YYYY-MM-DD", ',
                '"document_type": "discharge_summary|referral_letter|admission_note", ',
                '"primary_diagnosis": "diagnosis", "follow_up_required": true|false}}. ',
                'Return ONLY the JSON, no markdown, no explanation. ',
                'Text may be Dutch, Latin, or English.\\n\\n', full_text
            ),
            returnType => 'STRING'
        ) AS ai_response
    FROM text_extracted
    WHERE full_text IS NOT NULL
),
structured AS (
    SELECT
        filename,
        content_hash,
        try_cast(
            from_json(
                regexp_replace(ai_response, '```(?:json)?\\\\s*|\\\\s*```', ''),
                'struct<patient_ref:string,document_date:string,document_type:string,primary_diagnosis:string,follow_up_required:boolean>'
            ) AS struct<patient_ref:string,document_date:string,document_type:string,primary_diagnosis:string,follow_up_required:boolean>
        ) AS extracted
    FROM ai_extraction
)
SELECT
    filename,
    content_hash,
    'v1' AS prompt_version,
    struct(
        pseudonymise(extracted.patient_ref) AS patient_ref,
        try_to_date(extracted.document_date) AS document_date,
        extracted.document_type AS document_type,
        extracted.primary_diagnosis AS primary_diagnosis,
        extracted.follow_up_required AS follow_up_required
    ) AS extracted,
    current_timestamp() AS extracted_at
FROM structured
""")

bronze_df.createOrReplaceTempView("bronze_documents")

print(f"Bronze: {bronze_df.count()} documents processed")
display(spark.sql("SELECT COUNT(*) AS total_docs, COUNT(DISTINCT content_hash) AS unique_docs FROM bronze_documents"))

Bronze: 5 documents processed


total_docs,unique_docs
5,5


In [0]:
%sql
CREATE OR REPLACE FUNCTION strip_noise(text STRING)
RETURNS STRING
RETURN regexp_replace(
    regexp_replace(
        text,
        'Vertrouwelijk\\s*-\\s*[\\w\\s]+\\s*-\\s*Pagina \\d+ van \\d+',
        ''
    ),
    '\\n{3,}', '\\n\\n'
);

In [0]:
# Silver: Unnest struct, add business logic, apply cleaning

silver_df = spark.sql("""
SELECT
    extracted.patient_ref AS patient_ref,
    extracted.document_date AS document_date,
    lower(trim(extracted.document_type)) AS document_type,
    strip_noise(extracted.primary_diagnosis) AS primary_diagnosis,
    extracted.follow_up_required AS follow_up_required,
    datediff(current_date(), extracted.document_date) AS document_age_days,
    CASE 
        WHEN extracted.follow_up_required = true 
        THEN date_add(extracted.document_date, 30)
        ELSE null 
    END AS follow_up_due_date,
    extracted_at AS processed_at
FROM bronze_documents
WHERE prompt_version = 'v1'
""")

silver_df.createOrReplaceTempView("silver_documents")

print(f"Silver: {silver_df.count()} documents")
display(silver_df)

Silver: 5 documents


patient_ref,document_date,document_type,primary_diagnosis,follow_up_required,document_age_days,follow_up_due_date,processed_at
57a181cf1a85e84934929f3e7fc773f7,2025-01-10,discharge_summary,Urosepsis secondary to complicated urinary tract infection,true,421,2025-02-09,2026-03-07T22:02:35.806Z
f8fc08a55c80cf5f445e9fda0b094a37,2024-06-29,nursing_progress_note,not explicitly stated,true,616,2024-07-29,2026-03-07T22:02:35.806Z
759f7005dd73d9c7d1cd2dc7d771e9eb,2025-01-22,nursing_progress_note,diabetes,true,409,2025-02-21,2026-03-07T22:02:35.806Z
3d2bbfc4f200c7310929e6db48c0c64f,2024-09-08,admission_note,Life-threatening asthma exacerbation,true,545,2024-10-08,2026-03-07T22:02:35.806Z
fbe66ddf3a2fce15b7ddaf13aad7624e,2024-08-14,nursing_progress_note,right hip hemiarthroplasty,true,570,2024-09-13,2026-03-07T22:02:35.806Z


In [0]:
# Monitor null rates to detect LLM extraction failures

bronze = spark.table("bronze_documents")
total = bronze.count()

if total == 0:
    print("No documents processed yet")
else:
    null_patient_refs = bronze.filter("extracted.patient_ref IS NULL").count()
    null_rate = null_patient_refs / total
    
    print(f"Total: {total} | Null patient_ref rate: {null_rate:.1%}")
    
    if null_rate > 0.05:
        raise Exception(f"Extraction failure rate {null_rate:.1%} exceeds threshold")
    else:
        print("Quality check passed")

Total: 5 | Null patient_ref rate: 0.0%
Quality check passed
